##**RAG-Based PDF Chatbot with Conversational Context Adaptation**

**Problem Statement**

Develop a chatbot that answers user questions based on the content of uploaded PDF documents.

The system must use Retrieval-Augmented Generation (RAG) — meaning answers must be grounded in information retrieved from the PDF, not generated from model memory alone.

The chatbot must also maintain conversational context across multiple turns, so that follow-up questions are understood in relation to previous questions and answers within the same session.

Requirements

• The system must accept one or more PDF files as input.

• Answers must be derived from the content of the uploaded PDFs, not from the model's pre
trained knowledge.

• The chatbot must retain conversation history within a session and use it when answering
follow-up questions.

• Only open-source models or free-tier APIs may be used — no paid commercial model APIs.

• The solution must run in Google Colab or the Antigravity IDE without additional local setup.

Deliverables

• Project pipeline/workflow diagram — to be submitted to and verified by a senior before any
coding begins.

• A working, runnable notebook implementing the complete RAG chatbot.

• A README explaining how to run the project from scratch.

##Configuration & Imports

In [1]:
# ==========================================================
# Configuration
# ==========================================================

# ---------- Chunking ----------
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

# ---------- Embedding Model ----------
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
#EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ---------- Vector Search ----------
SEARCH_TYPE = "similarity"
TOP_K = 8

# ---------- LLM ----------
LLM_PROVIDER = "groq"
LLM_MODEL = "llama-3.3-70b-versatile"
TEMPERATURE = 0
MAX_HISTORY = 6
# ---------- Paths ----------
FAISS_INDEX_PATH = "faiss_index"

In [2]:
!pip install -q langchain-community pypdf langchain-text-splitters langchain langchain-huggingface sentence-transformers faiss-cpu langchain_groq

In [3]:
# Standard Library Imports
import os
import time
import json
import torch
from datetime import datetime
from typing import List
import pandas as pd
import numpy as np
#from google.colab import userdata
from huggingface_hub import login
# LangChain Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

c:\Users\Himanshu\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Himanshu\AppData\Local\Temp\ipykernel_10992\2267350221.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


##Document Loader

In [4]:
def load_documents(pdf_paths: List[str]):
    """
    Load one or more PDF files and return LangChain Document objects.

    Parameters
    ----------
    pdf_paths : List[str]
        List of PDF file paths.

    Returns
    -------
    List[Document]
        Combined list of documents from all PDFs.
    """
    validate_pdf_paths(pdf_paths)
    documents = []

    for pdf_path in pdf_paths:

        if not os.path.exists(pdf_path):
            print(f"❌ File not found: {pdf_path}")
            continue

        print(f"📄 Loading: {os.path.basename(pdf_path)}")

        loader = PyPDFLoader(pdf_path)

        pdf_docs = loader.load()

        # Remove empty pages
        pdf_docs = [
            doc for doc in pdf_docs
            if doc.page_content.strip()
        ]

        documents.extend(pdf_docs)

    print("\n========== Document Statistics ==========")

    print(f"Total PDFs Loaded : {len(pdf_paths)}")

    print(f"Total Pages       : {len(documents)}")

    validate_documents(documents)
    return documents

##Error Handling & Validation

In [5]:
def validate_pdf_paths(pdf_paths):
    """
    Validate uploaded PDF file paths.

    Parameters
    ----------
    pdf_paths : list

    Raises
    ------
    ValueError
    FileNotFoundError
    """

    if not pdf_paths:
        raise ValueError(
            "Please upload at least one PDF."
        )

    for path in pdf_paths:

        if not os.path.exists(path):
            raise FileNotFoundError(
                f"File not found: {path}"
            )

        if not path.lower().endswith(".pdf"):
            raise ValueError(
                f"Not a PDF file: {path}"
            )

In [6]:
def validate_documents(documents):
    """
    Ensure documents contain readable text.

    Parameters
    ----------
    documents : list

    Raises
    ------
    ValueError
    """

    if len(documents) == 0:

        raise ValueError(
            "No readable text found in the uploaded PDF."
        )

In [7]:
def validate_question(question):
    """
    Validate user question.

    Parameters
    ----------
    question : str

    Raises
    ------
    ValueError
    """

    if question is None:
        raise ValueError(
            "Question cannot be None."
        )

    if not question.strip():
        raise ValueError(
            "Please enter a valid question."
        )

In [8]:
def validate_retrieved_documents(documents):
    """
    Ensure retrieval returned documents.

    Parameters
    ----------
    documents : list

    Raises
    ------
    ValueError
    """

    if len(documents) == 0:

        raise ValueError(
            "No relevant information found in the uploaded PDF."
        )

In [9]:
def validate_vector_store(vector_store):
    """
    Validate FAISS vector store.
    """

    if vector_store is None:

        raise ValueError(
            "Vector store has not been initialized."
        )

In [10]:
def safe_ask(pipeline, question):
    """
    Safely execute the RAG pipeline.
    """

    try:

        return pipeline.ask(question)

    except Exception as error:

        return {
            "answer": str(error),
            "sources": "No sources available."
        }

##Uploading files here

In [11]:
#from google.colab import files

#uploaded = files.upload()

#pdf_paths = list(uploaded.keys())


## We can use this manual uploading in the session




In [16]:
documents = load_documents([r"C:\Users\Himanshu\Pythonscript_ckp\LLM(GenAI) Projects\RAG-Document-chatbot\uploaded_pdfs\Report.pdf"])

📄 Loading: Report.pdf

========== Document Statistics ==========
Total PDFs Loaded : 1
Total Pages       : 23


In [17]:
documents = load_documents([r"uploaded_pdfs\Report.pdf"])

📄 Loading: Report.pdf

========== Document Statistics ==========
Total PDFs Loaded : 1
Total Pages       : 23


##Document Chunking

In [19]:
def chunk_documents(documents):
    """
    Split LangChain documents into smaller chunks.

    Parameters
    ----------
    documents : List[Document]
        Documents returned by load_documents().

    Returns
    -------
    List[Document]
        Chunked documents preserving metadata.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ]
    )

    chunks = text_splitter.split_documents(documents)

    print("\n========== Chunk Statistics ==========")
    print(f"Original Documents : {len(documents)}")
    print(f"Total Chunks       : {len(chunks)}")
    print(f"Chunk Size         : {CHUNK_SIZE}")
    print(f"Chunk Overlap      : {CHUNK_OVERLAP}")

    return chunks

In [20]:
chunks = chunk_documents(documents)


========== Chunk Statistics ==========
Original Documents : 23
Total Chunks       : 73
Chunk Size         : 500
Chunk Overlap      : 100


##Chunk Statistics

In [21]:
def chunk_statistics(chunks):
    """
    Display statistics about generated chunks.
    """

    lengths = [len(chunk.page_content) for chunk in chunks]

    print("\n========== Chunk Statistics ==========")

    print(f"Total Chunks    : {len(chunks)}")
    print(f"Average Length  : {sum(lengths)/len(lengths):.2f}")
    print(f"Minimum Length  : {min(lengths)}")
    print(f"Maximum Length  : {max(lengths)}")

In [22]:
chunk_statistics(chunks)


========== Chunk Statistics ==========
Total Chunks    : 73
Average Length  : 404.40
Minimum Length  : 55
Maximum Length  : 498


##Inspect Chunks

In [23]:
def inspect_chunks(chunks, start=0, end=3):
    """
    Display selected chunks with metadata.
    """

    for i in range(start, min(end, len(chunks))):

        print("=" * 80)

        print(f"Chunk {i+1}")

        print("-" * 80)

        print("Source :", chunks[i].metadata.get("source"))

        print("Page   :", chunks[i].metadata.get("page", 0) + 1)

        print()

        print(chunks[i].page_content)

        print()

In [24]:
inspect_chunks(chunks)

Chunk 1
--------------------------------------------------------------------------------
Source : uploaded_pdfs\Report.pdf
Page   : 1

1  
  
     GUJARAT TECHNOLOGICAL UNIVERSITY   
Chandkheda, Ahmedabad 
Affiliated 
                                                                               
   C. K. Pithawalla College of Engineering and Technology,   
Surat   
A   
Project Report   
On   
   
(Campus Navigation System)   
   
     DESIGN ENGINEERING – II A (3150001)   
B. E. III, Semester – V    
(Computer Engineering)   
  
Submitted by:   
  
Sr. No.   Name of Student   Enrolment No

Chunk 2
--------------------------------------------------------------------------------
Source : uploaded_pdfs\Report.pdf
Page   : 1

(Computer Engineering)   
  
Submitted by:   
  
Sr. No.   Name of Student   Enrolment No   
 01   Himanshu Mahendra Mali   230090107081  
 02   Devansh Dharmesh Parekh   230090107111  
 03   Kunjan Jitendrakumar Patel   230090107144  
 04   Yash Harsh Patel    2300

##Chunk Search

In [25]:
def search_chunks(chunks, keyword):
    """
    Search all chunks containing a keyword.
    """

    found = False

    for chunk in chunks:

        if keyword.lower() in chunk.page_content.lower():

            found = True

            print("="*80)

            print(chunk.metadata)

            print(chunk.page_content)

            print()

    if not found:
        print("No matching chunk found.")

In [26]:
search_chunks(
    chunks,
    "Objective of the Project"
)

{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-10T19:59:57+05:30', 'title': 'DE.pdf', 'author': 'kunjan patel', 'moddate': '2026-04-10T19:59:57+05:30', 'source': 'uploaded_pdfs\\Report.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2'}
2  
  
Table of Contents 
   
ACKNOWLEDGMENT 3 
ABSTRACT 4 
1. Introduction  5 
1.1 Background 5 
      1.2 Problem Statement 5 
      1.3 Objective of the Project  6 
      1.4 Importance of Domain 6 
      1.5 Target Community 6 
      1.6 Scope of the Project  7 
      1.7 Reason for Selecting this Domain 7 
2. Canvases 8 
      2.1 AEIOUS Summary 8 
      2.2 Mind Mapping 9 
      2.3 Empathy Mapping 10 
      2.4 Ideation Canvas 11 
      2.5 Product Development Canvas  12

{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-10T19:59:57+05:30', 'title': 'DE.pdf', 'author': 'kunjan patel', 'moddate': '2026-04-10T19:59:57+05:30', 'source': 'uploaded_pdfs

##Initialize Embedding Model

In [29]:
pip install python-dotenv

In [30]:
from dotenv import load_dotenv

def create_embedding_model():
    """
    Initialize the HuggingFace embedding model.

    Returns
    -------
    HuggingFaceEmbeddings
        Configured embedding model.
    """

    print("Loading Embedding Model...")
    print(f"Model : {EMBEDDING_MODEL}")

    # Login to Hugging Face using Colab Secret
    load_dotenv()

    hf_token = os.getenv("HF_TOKEN")

    if hf_token is None:
        raise ValueError("HF_TOKEN not found in .env file.")

    login(token=hf_token)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    embedding_model = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={
            "device": device
        },
        encode_kwargs={
            "normalize_embeddings": True
        }
    )

    print(f"Using device: {device}")
    print("Embedding model loaded successfully.\n")

    return embedding_model

In [31]:
embedding_model = create_embedding_model()

Loading Embedding Model...
Model : BAAI/bge-base-en-v1.5


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Himanshu\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Himanshu\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using device: cpu
Embedding model loaded successfully.



##Mid Testing the Model

In [32]:
def test_embedding_model(embedding_model):
    """
    Test the embedding model on a sample sentence.
    """

    sample = "Artificial Intelligence"

    embedding = embedding_model.embed_query(sample)

    print("Embedding Dimension :", len(embedding))

    print("First 10 Values\n")

    print(embedding[:10])

In [33]:
test_embedding_model(embedding_model)

Embedding Dimension : 768
First 10 Values

[0.017266681417822838, 0.021780455484986305, 0.0006874478422105312, 0.02327672205865383, 0.03359045833349228, 0.042852018028497696, 0.02988690510392189, 0.02573092095553875, -0.012107516638934612, -0.02564956247806549]


##Compare Two Sentences

In [34]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [35]:
def compare_sentences(embedding_model, sentence1, sentence2):
    """
    Compare semantic similarity between two sentences.
    """

    emb1 = embedding_model.embed_query(sentence1)

    emb2 = embedding_model.embed_query(sentence2)

    similarity = cosine_similarity(
        [emb1],
        [emb2]
    )[0][0]

    print("Sentence 1")

    print(sentence1)

    print()

    print("Sentence 2")

    print(sentence2)

    print()

    print(f"Cosine Similarity : {similarity:.4f}")

In [36]:
compare_sentences(
    embedding_model,
    "Artificial Intelligence",
    "AI"
)

Sentence 1
Artificial Intelligence

Sentence 2
AI

Cosine Similarity : 0.8868


##Display Embedding Information

In [37]:
def embedding_info():
    """
    Display current embedding configuration.
    """

    print("\n========== Embedding Configuration ==========")

    print(f"Model : {EMBEDDING_MODEL}")

    print("Normalization : Enabled")

    print("Vector Database : FAISS")
embedding_info()


========== Embedding Configuration ==========
Model : BAAI/bge-base-en-v1.5
Normalization : Enabled
Vector Database : FAISS


##Vector Database (FAISS)

In [38]:
##Build Vector Store
def build_vector_store(chunks, embedding_model):
    """
    Create a FAISS vector store from document chunks.

    Parameters
    ----------
    chunks : List[Document]
        Chunked documents.

    embedding_model : HuggingFaceEmbeddings
        Embedding model used to generate vector representations.

    Returns
    -------
    FAISS
        Initialized FAISS vector store.
    """

    print("\nCreating FAISS Vector Store...")

    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    print("FAISS Vector Store created successfully.")

    return vector_store

##Save Vector Store

In [39]:
def save_vector_store(vector_store):
    """
    Save the FAISS vector store to disk.
    """

    vector_store.save_local(FAISS_INDEX_PATH)

    print(f"Vector Store saved to '{FAISS_INDEX_PATH}'")

In [40]:
def load_vector_store(embedding_model):
    """
    Load an existing FAISS vector store.

    Parameters
    ----------
    embedding_model : HuggingFaceEmbeddings

    Returns
    -------
    FAISS
    """

    if not os.path.exists(FAISS_INDEX_PATH):

        raise FileNotFoundError(
            f"No FAISS index found at '{FAISS_INDEX_PATH}'."
        )

    print("Loading FAISS Vector Store...")

    vector_store = FAISS.load_local(
        FAISS_INDEX_PATH,
        embedding_model,
        allow_dangerous_deserialization=True
    )

    print("Vector Store loaded successfully.")

    return vector_store

In [41]:
def vector_store_info(vector_store):
    """
    Display information about the FAISS vector store.
    """

    print("\n========== Vector Store ==========")

    print(f"Indexed Chunks : {vector_store.index.ntotal}")

    print(f"Index Path     : {FAISS_INDEX_PATH}")

In [43]:
def test_vector_store(vector_store, query, k=3):
    """
    Perform a quick similarity search.
    """

    print(f"\nQuery: {query}\n")

    results = vector_store.similarity_search(
        query=query,
        k=k
    )

    for i, doc in enumerate(results, start=1):

        print("=" * 80)

        print(f"Result {i}")

        print(f"Page : {doc.metadata.get('page', 0)+1}")

        print()

        print(doc.page_content[:300])

        print()

In [44]:
## Quick test
vector_store = build_vector_store(
    chunks,
    embedding_model
)

save_vector_store(vector_store)

vector_store_info(vector_store)

test_vector_store(
    vector_store,
    "What are the objectives of the project?"
)


Creating FAISS Vector Store...
FAISS Vector Store created successfully.
Vector Store saved to 'faiss_index'

========== Vector Store ==========
Indexed Chunks : 73
Index Path     : faiss_index

Query: What are the objectives of the project?

Result 1
Page : 6

6  
  
1.3 Objective of the Project 
The main objective of the project is to design and develop a Campus Navigation 
System that can:  
• Provide accurate and interactive navigation across the campus.  
• Allow users to search for classrooms, labs, or offices quickly.  
• Use GPS and digital maps fo

Result 2
Page : 2

2  
  
Table of Contents 
   
ACKNOWLEDGMENT 3 
ABSTRACT 4 
1. Introduction  5 
1.1 Background 5 
      1.2 Problem Statement 5 
      1.3 Objective of the Project  6 
      1.4 Importance of Domain 6 
      1.5 Target Community 6 
      1.6 Scope of the Project  7 
      1.7 Reason for Selecting th

Result 3
Page : 4

convenient and reliable for day-to-day use.  
This project has been developed following the Desi

##Retriever Engine

In [45]:
def create_retriever(vector_store):
    """
    Create a document retriever from the FAISS vector store.

    Parameters
    ----------
    vector_store : FAISS
        Initialized FAISS vector store.

    Returns
    -------
    VectorStoreRetriever
        Configured retriever.
    """

    print("\nInitializing Retriever...")

    retriever = vector_store.as_retriever(
        search_type=SEARCH_TYPE,
        search_kwargs={
            "k": TOP_K
        }
    )

    print("Retriever initialized successfully.")
    validate_vector_store(vector_store)
    return retriever
retriever = create_retriever(vector_store)


Initializing Retriever...
Retriever initialized successfully.


##Retrieve Context

In [46]:
def retrieve_context(retriever, query):
    """
    Retrieve the most relevant document chunks.

    Parameters
    ----------
    retriever : VectorStoreRetriever

    query : str
        User question.

    Returns
    -------
    list
        Retrieved LangChain Document objects.
    """

    documents = retriever.invoke(query)

    return documents

In [47]:
docs = retrieve_context(
    retriever,
    "What are the objectives of the project?"
)

In [48]:
def debug_retrieval(retriever, query):
    """
    Display retrieved chunks for debugging.
    """

    documents = retrieve_context(
        retriever,
        query
    )

    print("\n")
    print("=" * 100)
    print(f"Query : {query}")
    print("=" * 100)

    for i, doc in enumerate(documents, start=1):

        metadata = doc.metadata

        print(f"\nChunk {i}")

        print("-" * 80)

        print(f"Source : {metadata.get('source')}")

        print(f"Page   : {metadata.get('page',0)+1}")

        print()

        print(doc.page_content)

        print()

In [49]:
debug_retrieval(
    retriever,
    "What are the objectives of the project?"
)



Query : What are the objectives of the project?

Chunk 1
--------------------------------------------------------------------------------
Source : uploaded_pdfs\Report.pdf
Page   : 6

6  
  
1.3 Objective of the Project 
The main objective of the project is to design and develop a Campus Navigation 
System that can:  
• Provide accurate and interactive navigation across the campus.  
• Allow users to search for classrooms, labs, or offices quickly.  
• Use GPS and digital maps for outdoor navigation.  
• Provide QR code-based indoor navigation where GPS signals may not work. 
• Enhance user experience by reducing time and confusion.  
 
1.4 Importance of Domain


Chunk 2
--------------------------------------------------------------------------------
Source : uploaded_pdfs\Report.pdf
Page   : 2

2  
  
Table of Contents 
   
ACKNOWLEDGMENT 3 
ABSTRACT 4 
1. Introduction  5 
1.1 Background 5 
      1.2 Problem Statement 5 
      1.3 Objective of the Project  6 
      1.4 Importance of

##Search with Scores

In [50]:
def search_with_scores(vector_store, query, k=TOP_K):
    """
    Retrieve documents with similarity scores.
    """

    results = vector_store.similarity_search_with_score(
        query=query,
        k=k
    )

    print("\n")
    print("=" * 100)
    print(f"Query : {query}")
    print("=" * 100)

    for i, (doc, score) in enumerate(results, start=1):

        print(f"\nResult {i}")

        print("-" * 80)

        print(f"Score  : {score:.4f}")

        print(f"Page   : {doc.metadata.get('page',0)+1}")

        print()

        print(doc.page_content[:400])

        print()

In [51]:
search_with_scores(
    vector_store,
    "What are the objectives of the project?"
)



Query : What are the objectives of the project?

Result 1
--------------------------------------------------------------------------------
Score  : 0.5525
Page   : 6

6  
  
1.3 Objective of the Project 
The main objective of the project is to design and develop a Campus Navigation 
System that can:  
• Provide accurate and interactive navigation across the campus.  
• Allow users to search for classrooms, labs, or offices quickly.  
• Use GPS and digital maps for outdoor navigation.  
• Provide QR code-based indoor navigation where GPS signals may not work. 
•


Result 2
--------------------------------------------------------------------------------
Score  : 0.6379
Page   : 2

2  
  
Table of Contents 
   
ACKNOWLEDGMENT 3 
ABSTRACT 4 
1. Introduction  5 
1.1 Background 5 
      1.2 Problem Statement 5 
      1.3 Objective of the Project  6 
      1.4 Importance of Domain 6 
      1.5 Target Community 6 
      1.6 Scope of the Project  7 
      1.7 Reason for Selecting this Domain 

##Format Context

In [52]:
def format_context(documents):
    """
    Convert retrieved documents into a formatted context string.

    Parameters
    ----------
    documents : list
        Retrieved LangChain documents.

    Returns
    -------
    str
        Formatted context.
    """

    context = []

    for i, doc in enumerate(documents, start=1):

        metadata = doc.metadata

        source = metadata.get("source", "Unknown")

        page = metadata.get("page", 0) + 1

        context.append(
            f"""
========== Document {i} ==========
Source : {source}
Page   : {page}

{doc.page_content}
"""
        )

    return "\n".join(context)

In [53]:
docs = retrieve_context(
    retriever,
    "What are the objectives?"
)

context = format_context(docs)

print(context)


========== Document 1 ==========
Source : uploaded_pdfs\Report.pdf
Page   : 6

6  
  
1.3 Objective of the Project 
The main objective of the project is to design and develop a Campus Navigation 
System that can:  
• Provide accurate and interactive navigation across the campus.  
• Allow users to search for classrooms, labs, or offices quickly.  
• Use GPS and digital maps for outdoor navigation.  
• Provide QR code-based indoor navigation where GPS signals may not work. 
• Enhance user experience by reducing time and confusion.  
 
1.4 Importance of Domain


========== Document 2 ==========
Source : uploaded_pdfs\Report.pdf
Page   : 19

credentials, personal preferences, and saved locations. It also enables the system to 
provide a personalized experience by identifying different types of users like 
students, faculty, and administrators. 
By implementing authentication, the system becomes more secure, reliable, and 
suitable for real-world usage. 
 
8.2 Objectives of Authentication

##Test Retriever

In [54]:
def test_retriever(retriever, question):
    """
    Test the retriever and display formatted context.
    """

    documents = retrieve_context(
        retriever,
        question
    )

    print("\nRetrieved Chunks :", len(documents))

    print()

    print(format_context(documents))

In [55]:
test_retriever(
    retriever,
    "What are the objectives of the project?"
)


Retrieved Chunks : 8


========== Document 1 ==========
Source : uploaded_pdfs\Report.pdf
Page   : 6

6  
  
1.3 Objective of the Project 
The main objective of the project is to design and develop a Campus Navigation 
System that can:  
• Provide accurate and interactive navigation across the campus.  
• Allow users to search for classrooms, labs, or offices quickly.  
• Use GPS and digital maps for outdoor navigation.  
• Provide QR code-based indoor navigation where GPS signals may not work. 
• Enhance user experience by reducing time and confusion.  
 
1.4 Importance of Domain


========== Document 2 ==========
Source : uploaded_pdfs\Report.pdf
Page   : 2

2  
  
Table of Contents 
   
ACKNOWLEDGMENT 3 
ABSTRACT 4 
1. Introduction  5 
1.1 Background 5 
      1.2 Problem Statement 5 
      1.3 Objective of the Project  6 
      1.4 Importance of Domain 6 
      1.5 Target Community 6 
      1.6 Scope of the Project  7 
      1.7 Reason for Selecting this Domain 7 
2. Canvases 8 
  

##Conversation Memory

In [56]:
class ConversationMemory:
    """
    Stores conversation history for the current chat session.
    """

    def __init__(self):
        """Initialize an empty conversation history."""
        self.history = []

    def add_message(self, role, content):
        """
        Add a message to the conversation history.

        Parameters
        ----------
        role : str
            'user' or 'assistant'

        content : str
            Message text.
        """

        self.history.append({
            "role": role,
            "content": content
        })

    def get_history(self, max_messages=None):
        """
        Return the complete conversation history, or a truncated version.

        Parameters
        ----------
        max_messages : int, optional
            The maximum number of messages to return from the end of the history.
            If None, the complete history is returned.

        Returns
        -------
        List[Dict]
            Conversation history.
        """
        if max_messages is None:
            return self.history
        return self.history[-max_messages:]

    def clear(self):
        """
        Clear the conversation history.
        """

        self.history = []

    def display(self):
        """
        Print the conversation history.
        """

        print("\n========== Conversation History ==========\n")

        for message in self.history:

            print(f"{message['role'].upper()}:")

            print(message["content"])

            print()

In [57]:
memory = ConversationMemory()

In [58]:
memory.add_message(
    "user",
    "What are the objectives of the project?"
)

memory.add_message(
    "assistant",
    "The objectives are..."
)

In [59]:
memory.display()


========== Conversation History ==========

USER:
What are the objectives of the project?

ASSISTANT:
The objectives are...



In [60]:
history = memory.get_history()

print(history)

[{'role': 'user', 'content': 'What are the objectives of the project?'}, {'role': 'assistant', 'content': 'The objectives are...'}]


In [61]:
memory.clear()

##LLM Engine & Prompt Builder

In [62]:
#from getpass import getpass



def initialize_llm():
    """
    Initialize the Groq LLM.

    Returns
    -------
    ChatGroq
        Configured LLM instance.
    """

    print("\nInitializing LLM...")
    
    groq_api_key = os.getenv("GROQ_API_KEY")


    llm = ChatGroq(
        model=LLM_MODEL,
        temperature=TEMPERATURE
    )

    print(f"Model : {LLM_MODEL}")
    print("LLM initialized successfully.")

    return llm
llm = initialize_llm()


Initializing LLM...
Model : llama-3.3-70b-versatile
LLM initialized successfully.


In [63]:
def build_prompt(context, history, question):
    """
    Build the prompt using conversation history and retrieved context.

    Parameters
    ----------
    context : str
        Retrieved document context.

    question : str
        User question.

    Returns
    -------
    str
        Complete prompt.
    """
    history_text = ""

    for message in history:
        history_text += f"{message['role'].capitalize()}: {message['content']}\n"

    prompt = f"""
You are an intelligent PDF Question Answering Assistant.

Your task is to answer ONLY from the supplied context.

Rules:
1. Read the complete context carefully.
2. Answer ONLY using the provided context.
3. Do NOT use your own knowledge.
4. If the answer is not available in the context, reply exactly:

I couldn't find relevant information in the uploaded PDF.

5. If multiple context documents contain relevant information, combine them into one complete answer.
6. Write the answer in clear bullet points whenever appropriate.
7. Do not mention "according to the context" or "based on the provided context".

==================================================
Conversation History
==================================================

{history_text}

==================================================
CONTEXT
==================================================

{context}

==================================================
QUESTION
==================================================

{question}

==================================================
ANSWER
==================================================
"""

    return prompt

In [64]:
def generate_answer(llm, prompt):
    """
    Generate an answer using the LLM.

    Parameters
    ----------
    llm : ChatGroq

    prompt : str

    Returns
    -------
    str
        Generated answer.
    """

    response = llm.invoke(prompt)

    return response.content

In [65]:
question = "What are the objectives of the project?"

prompt = build_prompt(
    context=context,
    history=history,
    question=question
)

answer = generate_answer(
    llm,
    prompt
)

print(answer)

The objectives of the project are:
* To design and develop a Campus Navigation System that can:
  • Provide accurate and interactive navigation across the campus.
  • Allow users to search for classrooms, labs, or offices quickly.
  • Use GPS and digital maps for outdoor navigation.
  • Provide QR code-based indoor navigation where GPS signals may not work.
  • Enhance user experience by reducing time and confusion.
 
Additionally, the objectives of the authentication module are:
* Secure Login: To ensure that only registered users can access the system.


In [66]:
def preview_prompt(prompt, max_length=1500):
    """
    Display the generated prompt.
    """

    print("=" * 80)
    print("Prompt Preview")
    print("=" * 80)

    print(prompt[:max_length])

    if len(prompt) > max_length:
        print("\n...")
preview_prompt(prompt)

Prompt Preview

You are an intelligent PDF Question Answering Assistant.

Your task is to answer ONLY from the supplied context.

Rules:
1. Read the complete context carefully.
2. Answer ONLY using the provided context.
3. Do NOT use your own knowledge.
4. If the answer is not available in the context, reply exactly:

I couldn't find relevant information in the uploaded PDF.

5. If multiple context documents contain relevant information, combine them into one complete answer.
6. Write the answer in clear bullet points whenever appropriate.
7. Do not mention "according to the context" or "based on the provided context".

Conversation History

User: What are the objectives of the project?
Assistant: The objectives are...


CONTEXT


========== Document 1 ==========
Source : uploaded_pdfs\Report.pdf
Page   : 6

6  
  
1.3 Objective of the Project 
The main objective of the project is to design and develop a Campus Navigation 
System that can:  
• Provide accurate and interactive navigatio

In [67]:
def test_llm(llm, context, question):
    """
    Test the complete prompt generation pipeline.
    """

    prompt = build_prompt(
        context=context,
        question=question
    )

    preview_prompt(prompt)

    answer = generate_answer(
        llm,
        prompt
    )

    print("\n")
    print("=" * 100)
    print("ANSWER")
    print("=" * 100)

    print(answer)

In [68]:
docs = retrieve_context(
    retriever,
    "What are the objectives of the project?"
)

context = format_context(docs)

# Replicating the functionality of test_llm with the correct arguments
# The original test_llm function in qoS1LorrPwqO was missing the 'history' argument.
# This modification addresses the error by directly calling the prompt building and generation steps.

prompt = build_prompt(
    context=context,
    history=history, # Pass the history argument
    question="What are the objectives of the project?"
)

preview_prompt(prompt)

answer = generate_answer(
    llm,
    prompt
)

print("\n")
print("=" * 100)
print("ANSWER")
print("=" * 100)

print(answer)

Prompt Preview

You are an intelligent PDF Question Answering Assistant.

Your task is to answer ONLY from the supplied context.

Rules:
1. Read the complete context carefully.
2. Answer ONLY using the provided context.
3. Do NOT use your own knowledge.
4. If the answer is not available in the context, reply exactly:

I couldn't find relevant information in the uploaded PDF.

5. If multiple context documents contain relevant information, combine them into one complete answer.
6. Write the answer in clear bullet points whenever appropriate.
7. Do not mention "according to the context" or "based on the provided context".

Conversation History

User: What are the objectives of the project?
Assistant: The objectives are...


CONTEXT


========== Document 1 ==========
Source : uploaded_pdfs\Report.pdf
Page   : 6

6  
  
1.3 Objective of the Project 
The main objective of the project is to design and develop a Campus Navigation 
System that can:  
• Provide accurate and interactive navigatio

##Source Citation Engine

In [69]:
from collections import defaultdict

In [70]:
def collect_sources(documents):
    """
    Collect unique source files and page numbers from retrieved documents.

    Parameters
    ----------
    documents : list
        Retrieved LangChain documents.

    Returns
    -------
    dict
        Dictionary containing source filenames and page numbers.
    """

    sources = defaultdict(set)

    for doc in documents:

        source = doc.metadata.get("source", "Unknown")

        page = doc.metadata.get("page", 0) + 1

        sources[source].add(page)

    return sources

In [71]:
def format_sources(documents):
    """
    Format retrieved document sources into a readable string.

    Parameters
    ----------
    documents : list

    Returns
    -------
    str
    """

    sources = collect_sources(documents)

    if not sources:
        return "No source information available."

    formatted = []

    for source, pages in sources.items():

        pages = sorted(pages)

        page_string = ", ".join(str(page) for page in pages)

        formatted.append(
            f"• {source} (Pages: {page_string})"
        )

    return "\n".join(formatted)

##RAG Pipeline (Orchestrator)

In [72]:
class RAGPipeline:
    """
    End-to-end Retrieval-Augmented Generation pipeline.
    """

    def __init__(self, retriever, llm, memory):
        """
        Initialize the pipeline.

        Parameters
        ----------
        retriever : VectorStoreRetriever
            Configured retriever.

        llm :
            Initialized language model.
        """

        self.retriever = retriever
        self.llm = llm
        self.memory = memory

    def ask(self, question):
      validate_question(question)
      # Save user message
      self.memory.add_message("user", question)

      # Retrieve context
      documents = retrieve_context(
          self.retriever,
          question
      )
      validate_retrieved_documents(documents)

      context = format_context(documents)

      # Get history
      history = self.memory.get_history(MAX_HISTORY)

      # Build prompt
      prompt = build_prompt(
          context=context,
          history=history,
          question=question
      )

      # Generate answer
      answer = generate_answer(
          self.llm,
          prompt
      )

      # Save assistant response
      self.memory.add_message(
          "assistant",
          answer
      )

      # Collect sources
      formatted_sources = format_sources(documents)

      return {

          "question": question,

          "answer": answer,

          "sources": formatted_sources,

          "retrieved_documents": documents

      }

In [73]:
memory = ConversationMemory()

pipeline = RAGPipeline(
    retriever=retriever,
    llm=llm,
    memory=memory
)

In [74]:
response = pipeline.ask(
    "What are the objectives of the project?"
)

print(response["answer"])

print(response["sources"])

* The main objective of the project is to design and develop a Campus Navigation System that can:
  * Provide accurate and interactive navigation across the campus.
  * Allow users to search for classrooms, labs, or offices quickly.
  * Use GPS and digital maps for outdoor navigation.
  * Provide QR code-based indoor navigation where GPS signals may not work.
  * Enhance user experience by reducing time and confusion.
* The project also involves designing a mobile application that integrates with GPS and custom campus maps, implementing a search feature, providing QR code scanning for indoor navigation, displaying step-by-step directions, and collecting user feedback to refine and improve the system.
• uploaded_pdfs\Report.pdf (Pages: 2, 4, 6, 7, 9, 19)


In [75]:
response = pipeline.ask(
    "What about its features?"
)
print(response["answer"])

* The system's core functions include:
  * Seamless direction assistance
  * Real-time guidance
  * Ability to avoid confusion for newcomers
  * Accessible design
  * Location search
  * Lists of common destinations
* Other features include:
  * Smart search function
  * Safety features, such as emergency location tracking
  * Personalized dashboard to view relevant information (e.g., upcoming classes, frequently used locations)
  * Route navigation feature to generate the shortest path across the campus
  * QR code-based indoor navigation
  * GPS and digital maps for outdoor navigation
  * Mobile-friendly design, allowing users to access navigation features on smartphones
* The system is built using components like:
  * GPS
  * HTML
  * CSS
  * User databases
  * Interactive mobile application
* Key highlights of the system's features include:
  * Easy to use
  * Mobile-friendly
  * Helpful navigation, reducing confusion and saving time
  * User-friendly interface, allowing first-time

In [76]:
memory.display()


========== Conversation History ==========

USER:
What are the objectives of the project?

ASSISTANT:
* The main objective of the project is to design and develop a Campus Navigation System that can:
  * Provide accurate and interactive navigation across the campus.
  * Allow users to search for classrooms, labs, or offices quickly.
  * Use GPS and digital maps for outdoor navigation.
  * Provide QR code-based indoor navigation where GPS signals may not work.
  * Enhance user experience by reducing time and confusion.
* The project also involves designing a mobile application that integrates with GPS and custom campus maps, implementing a search feature, providing QR code scanning for indoor navigation, displaying step-by-step directions, and collecting user feedback to refine and improve the system.

USER:
What about its features?

ASSISTANT:
* The system's core functions include:
  * Seamless direction assistance
  * Real-time guidance
  * Ability to avoid confusion for newcomers
  

In [77]:
def get_recent_history(self, max_messages=6):
    return self.history[-max_messages:]

In [78]:
get_recent_history(memory)

[{'role': 'user', 'content': 'What are the objectives of the project?'},
 {'role': 'assistant',
  'content': '* The main objective of the project is to design and develop a Campus Navigation System that can:\n  * Provide accurate and interactive navigation across the campus.\n  * Allow users to search for classrooms, labs, or offices quickly.\n  * Use GPS and digital maps for outdoor navigation.\n  * Provide QR code-based indoor navigation where GPS signals may not work.\n  * Enhance user experience by reducing time and confusion.\n* The project also involves designing a mobile application that integrates with GPS and custom campus maps, implementing a search feature, providing QR code scanning for indoor navigation, displaying step-by-step directions, and collecting user feedback to refine and improve the system.'},
 {'role': 'user', 'content': 'What about its features?'},
 {'role': 'assistant',
  'content': "* The system's core functions include:\n  * Seamless direction assistance\n 

##Chat interface

In [79]:
def start_chat():

    print("="*60)
    print("RAG Chatbot")
    print("Type 'exit' to quit.")
    print("="*60)

    while True:

        question = input("\nYou : ")

        if question.lower() in ["exit","quit"]:

            print("\nGoodbye!")

            break
        start_time = time.time()
        validate_question(question)
        #response = pipeline.ask(question)
        response = safe_ask(pipeline,question)

        print("\nAssistant :")

        print(response["answer"])

        print("\nSources:")

        print(response["sources"])
        end_time = time.time()

        response_time = end_time - start_time

        # Log the interaction
        logger.log(
            question=question,
            answer=response["answer"],
            response_time=response_time,
            retrieved_documents=response["retrieved_documents"],
            llm_model=LLM_MODEL,
            embedding_model=EMBEDDING_MODEL
        )

##Reset Manager

In [80]:
def reset_memory(memory):
    """
    Reset conversation memory.
    """

    memory.clear()

    print("Conversation memory cleared.")

##Evaluation & Logging

In [81]:
class RAGLogger:
    """
    Logger for storing RAG chatbot interactions and performance metrics.
    """

    def __init__(self):
        """Initialize an empty log list."""
        self.logs = []

    def log(
        self,
        question,
        answer,
        response_time,
        retrieved_documents,
        llm_model,
        embedding_model
    ):
        """
        Store one chatbot interaction.
        """

        sources = []

        for doc in retrieved_documents:
            sources.append({
                "source": doc.metadata.get("source", "Unknown"),
                "page": doc.metadata.get("page", 0) + 1
            })

        log_entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "question": question,
            "answer": answer,
            "response_time_seconds": round(response_time, 3),
            "retrieved_chunks": len(retrieved_documents),
            "sources": sources,
            "llm_model": llm_model,
            "embedding_model": embedding_model
        }

        self.logs.append(log_entry)

    def show_logs(self):
        """
        Display all logged interactions.
        """

        print("\n========== RAG Logs ==========\n")

        for i, log in enumerate(self.logs, start=1):

            print("=" * 80)
            print(f"Interaction {i}")
            print("=" * 80)

            print(f"Time          : {log['timestamp']}")
            print(f"Question      : {log['question']}")
            print(f"Response Time : {log['response_time_seconds']} sec")
            print(f"Chunks Used   : {log['retrieved_chunks']}")
            print(f"LLM           : {log['llm_model']}")
            print(f"Embedding     : {log['embedding_model']}")
            print(f"Sources       : {log['sources']}")

            print()

    def export_json(self, filename="rag_logs.json"):
        """
        Export logs to a JSON file.
        """

        with open(filename, "w") as file:
            json.dump(
                self.logs,
                file,
                indent=4
            )

        print(f"Logs exported to {filename}")

    def export_csv(self, filename="rag_logs.csv"):
        """
        Export logs to CSV.
        """

        rows = []

        for log in self.logs:

            rows.append({
                "Timestamp": log["timestamp"],
                "Question": log["question"],
                "Response Time": log["response_time_seconds"],
                "Chunks": log["retrieved_chunks"],
                "LLM": log["llm_model"],
                "Embedding": log["embedding_model"],
                "Soyce": log["sources"]
            })

        df = pd.DataFrame(rows)

        df.to_csv(
            filename,
            index=False
        )

        print(f"Logs exported to {filename}")

    def clear(self):
        """
        Clear all stored logs.
        """

        self.logs.clear()


def reset_logger(logger):
    """
    Reset the logger by clearing all stored logs.
    """

    logger.clear()

    print("Logger cleared.")


def reset_pipeline(memory, logger):
    """
    Reset both conversation memory and logger.
    """

    reset_memory(memory)
    reset_logger(logger)

    print("Pipeline reset completed.")


def reset_session(memory, logger):
    """
    Reset the current chatbot session.
    """

    reset_memory(memory)
    reset_logger(logger)

    print("\nSession reset successfully.")

In [82]:
logger = RAGLogger()

In [83]:
memory.display()


========== Conversation History ==========

USER:
What are the objectives of the project?

ASSISTANT:
* The main objective of the project is to design and develop a Campus Navigation System that can:
  * Provide accurate and interactive navigation across the campus.
  * Allow users to search for classrooms, labs, or offices quickly.
  * Use GPS and digital maps for outdoor navigation.
  * Provide QR code-based indoor navigation where GPS signals may not work.
  * Enhance user experience by reducing time and confusion.
* The project also involves designing a mobile application that integrates with GPS and custom campus maps, implementing a search feature, providing QR code scanning for indoor navigation, displaying step-by-step directions, and collecting user feedback to refine and improve the system.

USER:
What about its features?

ASSISTANT:
* The system's core functions include:
  * Seamless direction assistance
  * Real-time guidance
  * Ability to avoid confusion for newcomers
  

In [84]:
memory.clear()

In [85]:
start_chat()

RAG Chatbot
Type 'exit' to quit.

Assistant :
The main objective of the project is to design and develop a Campus Navigation System that can:
* Provide accurate and interactive navigation across the campus.
* Allow users to search for classrooms, labs, or offices quickly.
* Use GPS and digital maps for outdoor navigation.
* Provide QR code-based indoor navigation where GPS signals may not work.
* Enhance user experience by reducing time and confusion. 

Additionally, the project aims to:
* Apply technical knowledge with practical utility.
* Contribute towards making the college/university campus more convenient and reliable for day-to-day use.
* Provide a personalized experience by identifying different types of users.
* Ensure the system is secure, reliable, and suitable for real-world usage. 

The objectives of the authentication module are:
* Secure Login: To ensure that only registered users can access the system.

Sources:
• uploaded_pdfs\Report.pdf (Pages: 2, 3, 4, 6, 7, 19)

Ass

In [86]:
logger.show_logs()


========== RAG Logs ==========

Interaction 1
Time          : 2026-07-15 16:49:49
Question      : what is the objective of this project ?
Response Time : 1.174 sec
Chunks Used   : 8
LLM           : llama-3.3-70b-versatile
Embedding     : BAAI/bge-base-en-v1.5
Sources       : [{'source': 'uploaded_pdfs\\Report.pdf', 'page': 6}, {'source': 'uploaded_pdfs\\Report.pdf', 'page': 2}, {'source': 'uploaded_pdfs\\Report.pdf', 'page': 4}, {'source': 'uploaded_pdfs\\Report.pdf', 'page': 7}, {'source': 'uploaded_pdfs\\Report.pdf', 'page': 19}, {'source': 'uploaded_pdfs\\Report.pdf', 'page': 6}, {'source': 'uploaded_pdfs\\Report.pdf', 'page': 3}, {'source': 'uploaded_pdfs\\Report.pdf', 'page': 3}]

Interaction 2
Time          : 2026-07-15 16:50:05
Question      : users of this ?
Response Time : 1.133 sec
Chunks Used   : 8
LLM           : llama-3.3-70b-versatile
Embedding     : BAAI/bge-base-en-v1.5
Sources       : [{'source': 'uploaded_pdfs\\Report.pdf', 'page': 20}, {'source': 'uploaded_pdfs\\Rep

Logs Download part Execute at the end

In [90]:
logger.export_json()
logger.export_csv()
#files.download("rag_logs.csv")
#files.download("rag_logs.json")

Logs exported to rag_logs.json
Logs exported to rag_logs.csv


In [88]:
memory.display()


========== Conversation History ==========

USER:
what is the objective of this project ?

ASSISTANT:
The main objective of the project is to design and develop a Campus Navigation System that can:
* Provide accurate and interactive navigation across the campus.
* Allow users to search for classrooms, labs, or offices quickly.
* Use GPS and digital maps for outdoor navigation.
* Provide QR code-based indoor navigation where GPS signals may not work.
* Enhance user experience by reducing time and confusion. 

Additionally, the project aims to:
* Apply technical knowledge with practical utility.
* Contribute towards making the college/university campus more convenient and reliable for day-to-day use.
* Provide a personalized experience by identifying different types of users.
* Ensure the system is secure, reliable, and suitable for real-world usage. 

The objectives of the authentication module are:
* Secure Login: To ensure that only registered users can access the system.

USER:
user

In [89]:
reset_session(memory,logger)

Conversation memory cleared.
Logger cleared.

Session reset successfully.
